# Capítulo 3 - Exercício 3: Titanic



## Plano do exercício

1. Carregar os dados do Titanic.
2. Treinar um modelo para prever a coluna Survied com base nas outras colunas.
3. Avaliar a acurácia final.

In [1]:
import sys
import numpy as np
import pandas as pd

from packaging import version
import sklearn

assert sys.version_info >= (3, 7)
assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

np.random.seed(42)

# Titanic - Importando os dados

In [2]:
titanic_train = pd.read_csv("./titanic/train.csv")
titanic_test = pd.read_csv("./titanic/test.csv")
titanic_test_answer = pd.read_csv("./titanic/gender_submission.csv")

data = titanic_train.drop("Survived", axis=1)
label = titanic_train["Survived"]


## Definindo as features

In [3]:
num_features = ["Age", "SibSp", "Parch"]
cat_features = ["Pclass", "Sex", "Embarked"]

## Pipeline de pre-processamento

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessing = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

## Modelos
### RandomForest

In [5]:
from sklearn.ensemble import RandomForestClassifier

rnd_clf = Pipeline([
    ("preprocessing", preprocessing),
    ("classifier", RandomForestClassifier(random_state=42))
])

rnd_clf.fit(data,label)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'SibSp', 'Parch']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Pclass', 'Sex',
                                                   'Embarked'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

### Validação cruzada

In [6]:
from sklearn.model_selection import cross_val_score

cross_val_score(rnd_clf, data, label, cv=3, scoring="accuracy")

array([0.76430976, 0.81481481, 0.77441077])

### GridSearch

In [7]:
from sklearn.model_selection import GridSearchCV

param_grid = [
    {
        "classifier__n_estimators": [100, 200, 300],
        "classifier__max_depth": [None, 5, 10],
        "classifier__min_samples_split": [2, 5, 10],
        "classifier__min_samples_leaf": [1, 2, 4]
    }
]

grid_search = GridSearchCV(rnd_clf, param_grid, cv=3, scoring="accuracy")
grid_search.fit(data, label)

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['Age',
                                                                          'SibSp',
                                                                          'Parch']),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('onehot',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['Pclass',
                                                                          'Sex',
                                                                          'Embarked'])])),
                                       ('classifier',
                                        RandomForestClassifier(random_state=42))]),
             param_grid=[{'classifier__max_depth': [None, 5, 10],
                          'classifier__min_samples_leaf': [1, 2, 4],
                          'classifier__min_samples_split': [2, 5, 10],
                          'classifier__n_estimators': [100, 200, 300]}],
             scoring='accuracy')

In [8]:
print("Melhores parâmetros:", grid_search.best_params_)
print("Melhor acurácia média na validação cruzada:", grid_search.best_score_)

Melhores parâmetros: {'classifier__max_depth': 10, 'classifier__min_samples_leaf': 4, 'classifier__min_samples_split': 10, 'classifier__n_estimators': 100}
Melhor acurácia média na validação cruzada: 0.8260381593714926


In [9]:
from sklearn.metrics import accuracy_score

best_rnd_clf = grid_search.best_estimator_
test_rnd_clf = best_rnd_clf.predict(titanic_test)
accuracy_score(titanic_test_answer["Survived"], test_rnd_clf)

0.9066985645933014